# B1 · A complete logistic on the loans: pipeline and ablation study

**Type**: Implement with scikit-learn · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Build a classification pipeline on mixed data with missing values (numerics with missingness indicators, nominals of low and high cardinality, an ordinal),
measure in cross-validation the **contribution of each piece** with an ablation study and read the coefficients as odds ratios.

## Dataset

`ds.loans()` with the missing values (1500 applications, 19% defaulters): stratified 75/25 split. The missing values of `income` are **linked to the target**
(whoever did not repay the loan more often has the income missing), those of `job_tenure` to the contract type.

## Educational purpose

Put together what you learned in modules 2-4 and learn the **ablation** reasoning: remove one piece at a time and measure what it costs. You will discover
that the most useful information is not in the values, but in **which** values are missing.

## Task

1. Write `build(indicators, categoricals)` that returns a `Pipeline`: `FunctionTransformer(add_ratio)` (given) → `ColumnTransformer` → `LogisticRegression(C=1, max_iter=2000)`. Numerics `age, income, job_tenure, amount, ratio`: median imputation (with `add_indicator=indicators`) and standardization. If `categoricals` is true add: `city`, `contract_type`, `occupation` with most-frequent imputation and `OneHotEncoder(handle_unknown="ignore")`, and `education` with imputation, `OrdinalEncoder` in the order of `EDUCATION_ORDER` and scaler.
2. With `StratifiedKFold(5, shuffle=True, random_state=0)` on the training set compute the mean AUC of four variants and save it in `cv_auc` (a `pd.Series`, index: `numerics only`, `+ indicators`, `+ categoricals`, `all`), where `+ categoricals` means categoricals **without** indicators.
3. Save `indicators_gain` (difference between `+ indicators` and `numerics only`) and `categoricals_gain` (between `+ categoricals` and `numerics only`).
4. Train `all` on the training set and save `test_auc`. Then read the coefficients (the names: `final[1].get_feature_names_out()`): save in `missing_income_odds` the odds ratio of 'income missing versus present' **in original units** (exponential of the coefficient of `num__missingindicator_income` divided by the standard deviation with which the `StandardScaler` scaled that column) and in `top4_numeric` the names of the 4 features of the `num__` block with the largest coefficient in absolute value.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, OrdinalEncoder, StandardScaler

df = ds.loans()
X, y = df.drop(columns="defaulted"), df["defaulted"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
NUM = ["age", "income", "job_tenure", "amount", "ratio"]


def add_ratio(X):
    X = X.copy()
    X["ratio"] = X["amount"] / X["income"]              # NaN where income is missing: the imputer will handle it
    return X

## Your work

In [ ]:
def build(indicators, categoricals):
    ...

In [ ]:
cv_auc = pd.Series(dtype=float)
indicators_gain = None
categoricals_gain = None
cv_auc

In [ ]:
final = build(True, True).fit(X_tr, y_tr)
test_auc = None
missing_income_odds = None
top4_numeric = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Numerics: `make_pipeline(SimpleImputer(strategy="median", add_indicator=indicators), StandardScaler())`. Names and coefficients: `names = final[1].get_feature_names_out()` (the names are given by the `ColumnTransformer`, second step of the pipeline; `FunctionTransformer` does not expose them); `coef = pd.Series(final[-1].coef_[0], index=names)`. The scale: `scale = final[1].named_transformers_["num"][-1].scale_` (one per column of the `num__` block, in the same order as the names); position of the indicator: `list(names).index("num__missingindicator_income")`.'):
    check(list(cv_auc.index) == ["numerics only", "+ indicators", "+ categoricals", "all"], "four variants in the right order", "index: numerics only, + indicators, + categoricals, all")
    check(cv_auc["all"] > 0.80 and cv_auc["all"] == cv_auc.max(), "the complete variant is the best (AUC > 0.80)", "all = indicators + categoricals")
    check(close(indicators_gain, cv_auc["+ indicators"] - cv_auc["numerics only"]) and indicators_gain > 0.04,
          "the missingness indicators bring over 0.04 of AUC", "add_indicator=True in the SimpleImputer of the numerics")
    check(close(categoricals_gain, cv_auc["+ categoricals"] - cv_auc["numerics only"]) and 0.01 < categoricals_gain < indicators_gain,
          "the categoricals contribute less than the indicators", "categoricals_gain must be positive but lower than indicators_gain")
    check(abs(test_auc - cv_auc["all"]) < 0.06 and close(test_auc, roc_auc_score(y_te, final.predict_proba(X_te)[:, 1])), "test AUC consistent with the cross-validated one", "roc_auc_score on the test set with predict_proba")
    check(missing_income_odds is not None and missing_income_odds > 2, "having the income missing more than doubles the default odds", "exp(coef / scale) for the indicator column; the scale is final[1].named_transformers_['num'][-1].scale_")
    check(top4_numeric is not None and set(top4_numeric) >= {"num__ratio", "num__missingindicator_income"}, "ratio and the income-missing indicator are among the four heaviest numerics", "sort by |coef| the names starting with 'num__'")
done()

---
The commented solution is in `solutions/B_with_sklearn/B1_loans_pipeline_and_ablation.ipynb`. Open it after you pass the check, or after you have really tried.